# Stacking model A

### : xgb + rf + lr

## 1 데이터 불러오기

In [15]:
import numpy as np
import pandas as pd 

In [16]:
df=pd.read_csv(
    'hotel_preprocessed.csv',
    dtype={"agent": str})

display(df)

,source_row_id,group_id,split,hotel,arrival_date_month,meal,country,market_segment,distribution_channel,reserved_room_type,...,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,days_in_waiting_list,adr,total_of_special_requests,has_company,total_guests,total_nights,is_canceled
0,0,0,train,Resort Hotel,July,BB,PRT,Direct,Direct,C,...,0,0,0,0,0.00,0,0,2,0,0
1,1,1,train,Resort Hotel,July,BB,PRT,Direct,Direct,C,...,0,0,0,0,0.00,0,0,2,0,0
2,2,2,train,Resort Hotel,July,BB,GBR,Direct,Direct,A,...,0,0,0,0,75.00,0,0,1,1,0
3,3,3,train,Resort Hotel,July,BB,GBR,Corporate,Corporate,A,...,0,0,0,0,75.00,0,0,1,1,0
4,4,4,train,Resort Hotel,July,BB,GBR,Online TA,TA/TO,A,...,0,0,0,0,98.00,1,0,2,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119203,119385,84663,train,City Hotel,August,BB,BEL,Offline TA/TO,TA/TO,A,...,0,0,0,0,96.14,0,0,2,7,0
119204,119386,84664,train,City Hotel,August,BB,FRA,Online TA,TA/TO,E,...,0,0,0,0,225.43,2,0,3,7,0
119205,119387,84665,test,City Hotel,August,BB,DEU,Online TA,TA/TO,D,...,0,0,0,0,157.71,4,0,2,7,0
119206,119388,84666,train,City Hotel,August,BB,GBR,Online TA,TA/TO,A,...,0,0,0,0,104.40,0,0,2,7,0


## 2 데이터 분할

### 2.1 Train/Test 분리 

In [17]:
ID_COLS = ["source_row_id", "group_id", "split"]
TARGET = "is_canceled"

FEATURES = [
    col for col in df.columns
    if col not in ID_COLS + [TARGET]
]

train = df[df["split"] == "train"].copy()
test = df[df["split"] == "test"].copy()

X_train_all = train[FEATURES]
y_train_all = train[TARGET]
groups_train_all = train["group_id"]

X_test = test[FEATURES]
y_test = test[TARGET]

print("Train:", X_train_all.shape)
print("Test:", X_test.shape)

Train: (95367, 28)
Test: (23841, 28)


### 2.2 Train 내부 Validation 분리 

In [18]:
from sklearn.model_selection import StratifiedGroupKFold

sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=0
)

train_idx, valid_idx = next(
    sgkf.split(
        X_train_all,
        y_train_all,
        groups_train_all
    )
)

X_train = X_train_all.iloc[train_idx].copy()
y_train = y_train_all.iloc[train_idx].copy()
groups_train = groups_train_all.iloc[train_idx].copy()

X_valid = X_train_all.iloc[valid_idx].copy()
y_valid = y_train_all.iloc[valid_idx].copy()

print("Train:", len(X_train))
print("Validation:", len(X_valid))
print("Test:", len(X_test))

Train: 76291
Validation: 19076
Test: 23841


### 2.3 그룹 중복 확인

In [19]:
train_groups = set(groups_train)
valid_groups = set(
    groups_train_all.iloc[valid_idx]
)
test_groups = set(test["group_id"])

print("Train / Valid:", len(train_groups & valid_groups))
print("Train / Test:", len(train_groups & test_groups))
print("Valid / Test:", len(valid_groups & test_groups))

Train / Valid: 0
Train / Test: 0
Valid / Test: 0


## 3 모델별 전처리 Pipeline 생성

In [20]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

# 수치형 컬럼
numeric_cols = X_train.select_dtypes(
    include=np.number
).columns.tolist()

# 범주형 컬럼
categorical_cols = X_train.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

# 트리 모델용 수치형 전처리
numeric_tree = "passthrough"

# 로지스틱 회귀용 수치형 전처리
numeric_linear = Pipeline([
    ("scaler", StandardScaler())
])

# 범주형 전처리
categorical_pipe = Pipeline([
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        min_frequency=20
    ))
])

# XGBoost / Random Forest
tree_preprocessor = ColumnTransformer([
    ("num", numeric_tree, numeric_cols),
    ("cat", categorical_pipe, categorical_cols)
])

# Logistic Regression
linear_preprocessor = ColumnTransformer([
    ("num", numeric_linear, numeric_cols),
    ("cat", categorical_pipe, categorical_cols)
])

In [21]:
## min_frequency 비교 
# for col in ["agent", "country"]:
#     counts = X_train[col].value_counts()

#     print(f"\n[{col}]")
#     print(f"전체 범주 수: {len(counts)}")

#     for n in [20, 100]:
#         rare = counts[counts < n]

#         print(
#             f"{n}회 미만: {len(rare)}개 범주, "
#             f"전체 행의 {rare.sum() / len(X_train):.2%}"
#         )

## 4 Base Model 정의 

In [22]:
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

models = {
    "XGBoost": Pipeline([
        ("preprocessor", tree_preprocessor),
        ("model", XGBClassifier(
            n_estimators=300,
            max_depth=5,
            learning_rate=0.05,
            eval_metric="logloss",
            random_state=42,
            n_jobs=2
        ))
    ]),

    "Random Forest": Pipeline([
        ("preprocessor", tree_preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=200,
            max_depth=15,
            min_samples_leaf=3,
            random_state=42,
            n_jobs=2
        ))
    ]),

    "Logistic Regression": Pipeline([
        ("preprocessor", linear_preprocessor),
        ("model", LogisticRegression(
            max_iter=2000,
            random_state=42
        ))
    ])
}

## RandomizedSearchCV

In [23]:
param_distributions = {
    "XGBoost": {
        "model__n_estimators": [200, 300, 500],
        "model__max_depth": [3, 4, 5],
        "model__learning_rate": [0.03, 0.05, 0.1],
        "model__min_child_weight": [1, 3, 5],
        "model__subsample": [0.8, 1.0],
        "model__colsample_bytree": [0.8, 1.0],
        "model__reg_lambda": [1, 5, 10]
    },
    "Random Forest": {
        "model__n_estimators": [200, 300, 500],
        "model__max_depth": [10, 15, 20],
        "model__min_samples_leaf": [3, 5, 10, 20],
        "model__min_samples_split": [2, 10, 20],
        "model__max_features": ["sqrt", 0.5]
    },
    "Logistic Regression": {
        "model__C": [0.01, 0.1, 0.3, 1, 10]
    }
}

In [24]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedGroupKFold

tuning_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

best_models = {}
tuning_results = []

for name, model in models.items():

    print(f"\n{name} 하이퍼파라미터 튜닝 시작")

    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_distributions[name],
        n_iter=20,
        scoring="roc_auc",
        cv=tuning_cv,
        n_jobs=-1,
        random_state=42,
        refit=True,
        verbose=1
    )

    search.fit(
        X_train,
        y_train,
        groups=groups_train
    )

    best_models[name] = search.best_estimator_

    tuning_results.append({
        "Model": name,
        "Best ROC-AUC": search.best_score_,
        "Best Params": search.best_params_
    })

    print("Best ROC-AUC:", search.best_score_)
    print("Best Params:", search.best_params_)

tuning_results_df = pd.DataFrame(tuning_results)
display(tuning_results_df)


XGBoost 하이퍼파라미터 튜닝 시작
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best ROC-AUC: 0.9317845025324702
Best Params: {'model__subsample': 1.0, 'model__reg_lambda': 1, 'model__n_estimators': 500, 'model__min_child_weight': 5, 'model__max_depth': 5, 'model__learning_rate': 0.1, 'model__colsample_bytree': 1.0}

Random Forest 하이퍼파라미터 튜닝 시작
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best ROC-AUC: 0.934110735651157
Best Params: {'model__n_estimators': 500, 'model__min_samples_split': 2, 'model__min_samples_leaf': 3, 'model__max_features': 0.5, 'model__max_depth': 20}

Logistic Regression 하이퍼파라미터 튜닝 시작


/opt/miniconda3/envs/agent/lib/python3.12/site-packages/sklearn/model_selection/_search.py:327: UserWarning: The total space of parameters 5 is smaller than n_iter=20. Running 5 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 5 folds for each of 5 candidates, totalling 25 fits
Best ROC-AUC: 0.8901453706244707
Best Params: {'model__C': 0.3}


,Model,Best ROC-AUC,Best Params
0,XGBoost,0.931785,"{'model__subsample': 1.0, 'model__reg_lambda':..."
1,Random Forest,0.934111,"{'model__n_estimators': 500, 'model__min_sampl..."
2,Logistic Regression,0.890145,{'model__C': 0.3}


## 5 그룹 기반 OOF 예측 생성

In [25]:
from sklearn.base import clone
from sklearn.model_selection import StratifiedGroupKFold

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# OOF 예측을 저장할 DataFrame
oof_pred = pd.DataFrame(
    index=np.arange(len(X_train)),
    columns=list(models.keys()),
    dtype=float
)

# 실제 반복 학습 
fitted_models = {}

for name, model in best_models.items():

    print(f"\n{name} OOF 학습 시작")

    for fold, (tr_idx, val_idx) in enumerate(
        cv.split(X_train, y_train, groups_train),
        start=1
    ):
        fold_model = clone(model) 

        fold_model.fit(
            X_train.iloc[tr_idx],
            y_train.iloc[tr_idx]
        )

        probabilities = fold_model.predict_proba(
            X_train.iloc[val_idx]
        )[:, 1]

        oof_pred.iloc[
            val_idx,
            oof_pred.columns.get_loc(name)
        ] = probabilities

        print(f"Fold {fold} 완료")

    # Validation/Test 예측용 전체 Train 학습
    final_model = clone(model)
    final_model.fit(X_train, y_train)

    fitted_models[name] = final_model

print("\nOOF 결측치:", oof_pred.isna().sum().sum())
display(oof_pred.head())


XGBoost OOF 학습 시작
Fold 1 완료
Fold 2 완료
Fold 3 완료
Fold 4 완료
Fold 5 완료

Random Forest OOF 학습 시작
Fold 1 완료
Fold 2 완료
Fold 3 완료
Fold 4 완료
Fold 5 완료

Logistic Regression OOF 학습 시작
Fold 1 완료
Fold 2 완료
Fold 3 완료
Fold 4 완료
Fold 5 완료

OOF 결측치: 0


,XGBoost,Random Forest,Logistic Regression
0,0.060792,0.478044,0.317002
1,0.005032,0.014598,0.008928
2,0.009070,0.008967,0.021825
3,0.015859,0.074128,0.048636
4,0.015859,0.074128,0.048636


## 6 Meta model 학습

In [26]:
from sklearn.linear_model import LogisticRegression

meta_model = LogisticRegression(
    max_iter=2000,
    random_state=42
)

meta_model.fit(
    oof_pred,
    y_train.to_numpy()
)

,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",2000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in 

## 7 모델 성능 및 비교


In [27]:
# Validation 데이터에 대한 Base Model 예측
valid_meta_X = pd.DataFrame({
    name: model.predict_proba(X_valid)[:, 1]
    for name, model in fitted_models.items()
})

# Meta Model로 최종 취소 확률 계산
stacking_valid_proba = meta_model.predict_proba(
    valid_meta_X
)[:, 1]

# 취소 여부 예측 (기준 0.5)
stacking_valid_pred = (
    stacking_valid_proba >= 0.5
).astype(int)

In [28]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)

print("Accuracy:",
      accuracy_score(y_valid, stacking_valid_pred))

print("F1:",
      f1_score(y_valid, stacking_valid_pred))

print("ROC-AUC:",
      roc_auc_score(y_valid, stacking_valid_proba))

print("PR-AUC:",
      average_precision_score(y_valid, stacking_valid_proba))

print("Brier Score:",
      brier_score_loss(y_valid, stacking_valid_proba))

Accuracy: 0.8616062067519397
F1: 0.8038632986627043
ROC-AUC: 0.9395177066989673
PR-AUC: 0.9134377639771145
Brier Score: 0.09718333329610783


## 8 단일 모델과 stacking 비교

In [29]:
results = []

# 단일 모델 평가
for name, model in fitted_models.items():

    proba = model.predict_proba(X_valid)[:, 1]
    pred = (proba >= 0.5).astype(int)

    results.append({
        "Model": name,
        "ROC-AUC": roc_auc_score(y_valid, proba),
        "PR-AUC": average_precision_score(y_valid, proba),
        "F1": f1_score(y_valid, pred),
        "Brier Score": brier_score_loss(y_valid, proba)
    })

# Stacking 평가
results.append({
    "Model": "Stacking A",
    "ROC-AUC": roc_auc_score(
        y_valid, stacking_valid_proba
    ),
    "PR-AUC": average_precision_score(
        y_valid, stacking_valid_proba
    ),
    "F1": f1_score(
        y_valid, stacking_valid_pred
    ),
    "Brier Score": brier_score_loss(
        y_valid, stacking_valid_proba
    )
})

results_df = pd.DataFrame(results)

display(
    results_df.sort_values(
        "ROC-AUC",
        ascending=False
    )
)

,Model,ROC-AUC,PR-AUC,F1,Brier Score
3,Stacking A,0.939518,0.913438,0.803863,0.097183
1,Random Forest,0.936999,0.909873,0.800119,0.096943
0,XGBoost,0.936149,0.909552,0.797849,0.097268
2,Logistic Regression,0.893641,0.854444,0.729460,0.123966


## 9 Test data 로 최종 평가

In [1]:
test_meta_X = pd.DataFrame({
    name: model.predict_proba(X_test)[:, 1]
    for name, model in fitted_models.items()
})

stacking_test_proba = meta_model.predict_proba(
    test_meta_X
)[:, 1]

stacking_test_pred = (
    stacking_test_proba >= 0.5
).astype(int)
print("Accuracy:",
      accuracy_score(y_test, stacking_test_pred))

print("Test ROC-AUC:",
      roc_auc_score(y_test, stacking_test_proba))

print("Test PR-AUC:",
      average_precision_score(y_test, stacking_test_proba))

print("Test F1:",
      f1_score(y_test, stacking_test_pred))

print("Test Brier:",
      brier_score_loss(y_test, stacking_test_proba))

NameError: name 'pd' is not defined

## 10 best model 저장

In [ ]:
import joblib

stacking_model = {
    "base_models": fitted_models,
    "meta_model": meta_model
}

joblib.dump(stacking_model, "best_stacking_modelA.joblib")